In [1]:
import pandas as pd
import numpy as np
import sympy as sp
import seaborn as sns

from scipy import stats

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_excel(r"C:\Users\Caue Almiron\Downloads\PBL_Fase_05\cidade_alfa_transporte_publico.xlsx")

In [3]:
df.head(5)

,id_coleta,id_veiculo,data_viagem,regiao,linha,tipo_veiculo,tempo_viagem_min,atraso_estimado_min,temperatura_c,chuva_mm,idade_frota_anos,indice_risco_falha,status_operacional,nivel_bateria_pct,qualidade_percebida,satisfacao_passageiro,linha_no_horario
0,1,VEI-2000,2026-05-02 10:00:00,Oeste,Linha 205 - Centro Expresso,Micro_Onibus,14.15,7.08,21.1,0.0,10,58,Normal,87.0,Regular,4,Sim
1,2,VEI-2001,2026-05-09 22:00:00,Leste,Linha 410 - Horizonte,Van_Compartilhada,21.08,9.74,28.7,2.1,9,43,Superlotado,75.2,Excelente,4,Nao
2,3,VEI-2002,2026-03-06 06:00:00,Centro,Linha 301 - Parque Central,Onibus_Convencional,22.66,0.15,24.5,0.0,2,53,Normal,58.1,Excelente,5,Sim
3,4,VEI-2003,2026-02-08 12:00:00,Oeste,Linha 210 - Jardim Alfa,Onibus_Autonomo,19.99,11.88,35.8,0.0,8,49,Normal,73.8,Boa,4,Nao
4,5,VEI-2004,2026-01-20 05:00:00,Norte,Linha 301 - Parque Central,Onibus_Autonomo,16.49,3.50,18.8,0.0,13,84,Falha_Mecanica,69.0,Boa,3,Sim


### 1. Conhecendo a base (estrutura, nulos, duplicatas)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5069 entries, 0 to 5068
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_coleta              5069 non-null   int64         
 1   id_veiculo             5069 non-null   object        
 2   data_viagem            5069 non-null   datetime64[ns]
 3   regiao                 5069 non-null   object        
 4   linha                  5044 non-null   object        
 5   tipo_veiculo           5069 non-null   object        
 6   tempo_viagem_min       5028 non-null   float64       
 7   atraso_estimado_min    5069 non-null   float64       
 8   temperatura_c          5069 non-null   float64       
 9   chuva_mm               5069 non-null   float64       
 10  idade_frota_anos       5069 non-null   int64         
 11  indice_risco_falha     5069 non-null   int64         
 12  status_operacional     5069 non-null   object        
 13  niv

In [5]:
df.isna().sum()

id_coleta                 0
id_veiculo                0
data_viagem               0
regiao                    0
linha                    25
tipo_veiculo              0
tempo_viagem_min         41
atraso_estimado_min       0
temperatura_c             0
chuva_mm                  0
idade_frota_anos          0
indice_risco_falha        0
status_operacional        0
nivel_bateria_pct         0
qualidade_percebida       0
satisfacao_passageiro     0
linha_no_horario          0
dtype: int64

In [6]:
df.duplicated(subset=["id_veiculo", "data_viagem"]).sum()

0

### 2. Limpeza (nulos e outliers) - antes de seguir

In [7]:
# imputa tempo_viagem_min pela mediana do grupo (tipo_veiculo + regiao)
df["tempo_viagem_min"] = df.groupby(["tipo_veiculo", "regiao"])["tempo_viagem_min"].transform(
    lambda x: x.fillna(x.median())
)

df["linha"] = df["linha"].fillna("Nao_identificada")

# flag de outlier via IQR, sem descartar a linha
q1, q3 = df["tempo_viagem_min"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_sup = q3 + 1.5 * iqr
df["viagem_anomala"] = df["tempo_viagem_min"] > limite_sup

In [8]:
df["viagem_anomala"].sum()
df.groupby("tipo_veiculo")["viagem_anomala"].mean()  # % de anomalias por tipo de veículo

tipo_veiculo
Micro_Onibus           0.005155
Onibus_Autonomo        0.002894
Onibus_Convencional    0.023842
Van_Compartilhada      0.000000
Name: viagem_anomala, dtype: float64

### 3. Estatística descritiva

In [9]:
### Kurt = Curtose - mede o grau de achatamento da curva e o peso das suas caudas. 
### Ela indica se os dados têm muitos ou poucos valores extremos (outliers) 
### em comparação com a distribuição normal.


### Skew = Assimetria - A skewness mede a falta de simetria dos dados em relação à média. 
### Ela mostra para qual lado a "cauda" da distribuição está se alongando.

In [10]:
# descritiva geral
desc = df[["tempo_viagem_min", "atraso_estimado_min", "indice_risco_falha",
           "nivel_bateria_pct", "satisfacao_passageiro"]].describe()

# assimetria e curtose (mede se a distribuição é enviesada/tem cauda pesada)
skew_kurt = df[["tempo_viagem_min", "atraso_estimado_min"]].agg(["skew", "kurt"])

# descritiva por grupo
desc_por_tipo = df.groupby("tipo_veiculo")[["tempo_viagem_min", "atraso_estimado_min"]].agg(["mean", "median", "std"])
desc_por_regiao = df.groupby("regiao")["satisfacao_passageiro"].agg(["mean", "std", "count"])

In [11]:
desc

,tempo_viagem_min,atraso_estimado_min,indice_risco_falha,nivel_bateria_pct,satisfacao_passageiro
count,5069.000000,5069.000000,5069.000000,5069.000000,5069.000000
mean,26.354397,5.957528,51.251529,64.693786,3.984415
std,15.245788,3.330636,22.879054,21.145283,0.903358
min,1.000000,0.000000,0.000000,20.000000,1.000000
25%,20.270000,3.590000,34.000000,48.100000,3.000000
50%,25.270000,5.910000,51.000000,65.300000,4.000000
75%,31.110000,8.220000,69.000000,82.300000,5.000000
max,371.560000,19.400000,100.000000,100.000000,5.000000


In [12]:
skew_kurt

,tempo_viagem_min,atraso_estimado_min
skew,12.844292,0.21388
kurt,234.600658,-0.26491


In [13]:
desc_por_tipo

tempo_viagem_min                    atraso_estimado_min  \
                                mean  median        std                mean   
tipo_veiculo                                                                  
Micro_Onibus               22.178376  20.570  19.404098            5.154244   
Onibus_Autonomo            24.738187  24.165   9.931798            5.755658   
Onibus_Convencional        31.619220  30.025  20.404212            6.707772   
Van_Compartilhada          23.095137  22.530   7.722138            5.659961   

                                      
                    median       std  
tipo_veiculo                          
Micro_Onibus         5.025  3.212558  
Onibus_Autonomo      5.710  3.301729  
Onibus_Convencional  6.670  3.337974  
Van_Compartilhada    5.330  3.087126

In [14]:
desc_por_regiao

,mean,std,count
regiao,,,
Centro,3.952062,0.911814,897
Leste,4.034151,0.872035,1142
Norte,3.976399,0.886549,1144
Oeste,3.987718,0.914284,977
Sul,3.960396,0.941326,909


### 4. Amostragem

In [15]:
# amostra aleatória simples
amostra_simples = df.sample(n=500, random_state=42)

# amostra estratificada por tipo_veiculo (mantém proporção)
amostra_estratificada = df.groupby("tipo_veiculo", group_keys=False).sample(frac=0.1, random_state=42)

# checagem: proporções devem bater
print(df["tipo_veiculo"].value_counts(normalize=True))
print(amostra_estratificada["tipo_veiculo"].value_counts(normalize=True))

tipo_veiculo
Onibus_Autonomo        0.545275
Onibus_Convencional    0.289603
Micro_Onibus           0.114816
Van_Compartilhada      0.050306
Name: proportion, dtype: float64
tipo_veiculo
Onibus_Autonomo        0.544379
Onibus_Convencional    0.289941
Micro_Onibus           0.114398
Van_Compartilhada      0.051282
Name: proportion, dtype: float64


### 5. Correlação

In [16]:
corr = df[["tempo_viagem_min", "atraso_estimado_min", "idade_frota_anos",
           "indice_risco_falha", "chuva_mm", "satisfacao_passageiro"]].corr(method="pearson")

# teste de significância de uma correlação específica (idade da frota x risco de falha)
r, p_valor = stats.pearsonr(df["idade_frota_anos"], df["indice_risco_falha"])
print(f"r={r:.3f}, p={p_valor:.5f}")
print(f"Correlação forte e significativa: quanto mais velha a frota, maior o risco de falha reportado.")

r=0.859, p=0.00000
Correlação forte e significativa: quanto mais velha a frota, maior o risco de falha reportado.


In [17]:
corr

,tempo_viagem_min,atraso_estimado_min,idade_frota_anos,indice_risco_falha,chuva_mm,satisfacao_passageiro
tempo_viagem_min,1.000000,0.201078,-0.004426,-0.003300,0.266848,-0.077996
atraso_estimado_min,0.201078,1.000000,0.352332,0.299458,0.178059,-0.360990
idade_frota_anos,-0.004426,0.352332,1.000000,0.859214,0.001956,-0.249093
indice_risco_falha,-0.003300,0.299458,0.859214,1.000000,0.005424,-0.249378
chuva_mm,0.266848,0.178059,0.001956,0.005424,1.000000,-0.092381
satisfacao_passageiro,-0.077996,-0.360990,-0.249093,-0.249378,-0.092381,1.000000


### 6. Teste de Hipótese

In [18]:
# H0: tempo médio de viagem é igual entre Onibus_Autonomo e Onibus_Convencional
autonomo = df.loc[df["tipo_veiculo"] == "Onibus_Autonomo", "tempo_viagem_min"]
convencional = df.loc[df["tipo_veiculo"] == "Onibus_Convencional", "tempo_viagem_min"]

t_stat, p_valor = stats.ttest_ind(autonomo, convencional, equal_var=False)

# ANOVA: satisfação difere entre regiões?
grupos = [g["satisfacao_passageiro"].values for _, g in df.groupby("regiao")]
f_stat, p_valor_anova = stats.f_oneway(*grupos)

# qui-quadrado: status_operacional está associado a qualidade_percebida?
tabela = pd.crosstab(df["status_operacional"], df["qualidade_percebida"])
chi2, p_valor_chi2, dof, esperado = stats.chi2_contingency(tabela)

In [19]:
print(f"t_stat={t_stat:.4f}, p_valor={p_valor:.10f}")
print(f"f_stat={f_stat:.4f}, p_valor_anova={p_valor_anova:.4f}")
print(f"chi2={chi2:.4f}, p_valor_chi2={p_valor_chi2:.2e}, dof={dof}")

t_stat=-12.1775, p_valor=0.0000000000
f_stat=1.3398, p_valor_anova=0.2525
chi2=1465.7590, p_valor_chi2=9.19e-307, dof=12


**Interpretação dos testes de hipótese:**

- **Teste t (autônomo x convencional):** p < 0.05 → rejeita H0. O tempo médio de viagem difere significativamente entre os dois tipos de veículo (o ônibus autônomo é, em média, mais rápido).
- **ANOVA (satisfação por região):** p = 0.2525 > 0.05 → não rejeita H0. A satisfação do passageiro **não** difere significativamente entre as regiões — indício de que o problema é operacional (frota, status), não geográfico.
- **Qui-quadrado (status operacional x qualidade percebida):** p << 0.05 → rejeita H0. Há associação forte entre o status operacional e a qualidade percebida (viagens com `Falha_Mecanica` concentram muito mais avaliações `Ruim`/`Regular`, como mostra a tabela abaixo).

In [20]:
tabela

qualidade_percebida,Boa,Excelente,Regular,Ruim
status_operacional,,,,
Atraso_Critico,254,258,80,12
Falha_Mecanica,146,32,244,179
Manutencao,336,365,119,7
Normal,818,1034,274,50
Superlotado,280,85,324,172


### 7. Limites, derivadas e integrais

In [21]:
# série agregada: atraso médio por dia do ano
serie_diaria = df.groupby(df["data_viagem"].dt.date)["atraso_estimado_min"].mean().reset_index()
serie_diaria["dia_num"] = range(len(serie_diaria))

# ajusta uma função polinomial (grau 2) ao comportamento do atraso ao longo do tempo
coef = np.polyfit(serie_diaria["dia_num"], serie_diaria["atraso_estimado_min"], deg=2)
x = sp.symbols("x")
f = coef[0]*x**2 + coef[1]*x + coef[2]

# derivada: taxa de variação do atraso médio (piorando ou melhorando?)
f_derivada = sp.diff(f, x)
taxa_no_dia_100 = f_derivada.subs(x, 100)

# limite: tendência do atraso quando x cresce indefinidamente
limite_longo_prazo = sp.limit(f, x, sp.oo)

# integral: "atraso acumulado" estimado no período observado (área sob a curva)
integral_total = sp.integrate(f, (x, 0, len(serie_diaria)))

In [22]:
serie_diaria

,data_viagem,atraso_estimado_min,dia_num
0,2026-01-01,5.766250,0
1,2026-01-02,6.347273,1
2,2026-01-03,5.356154,2
3,2026-01-04,6.049524,3
4,2026-01-05,5.320800,4
...,...,...,...
206,2026-07-26,5.545484,206
207,2026-07-27,5.628333,207
208,2026-07-28,6.625000,208
209,2026-07-29,6.125806,209


In [23]:
coef

array([-1.05610672e-05,  2.15878806e-03,  5.88166853e+00])

In [24]:
f_derivada

0.00215878805973649 - 2.11221343390958e-5*x

In [25]:
taxa_no_dia_100

4.65746258269048e-5

In [26]:
limite_longo_prazo

-oo

In [27]:
integral_total

1256.01778319672

**Cuidado com a extrapolação:** o limite dá `-oo` porque o coeficiente quadrático do ajuste é levemente negativo (`-1.06e-05`). Isso é um artefato de projetar a parábola para além do intervalo observado (211 dias) — não significa que o atraso realmente tenderá a menos infinito no mundo real. Útil para ilustrar o conceito de limite, mas não deve ser lido como previsão válida fora da janela de dados coletada.